# Cell 1 — Imports

In [1]:
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.signal import find_peaks

# Cell 2 — Config / Paths

In [2]:
simu_index = 102749

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "../.."))

csv_path = os.path.join(BASE_DIR, "Data", "parameter.csv")
variation_dir = os.path.join(BASE_DIR, "Data", "variation")
file_path = os.path.join(variation_dir, f"simu_{simu_index}.s4p")

print(file_path)
print(os.path.exists(file_path))

/home/tekb/Master_Thesis_KB/CodesAndData/Data/variation/simu_102749.s4p
True


# Cell 3 — Constants

In [3]:
Z0 = 50
EPS0 = 8.854e-12
C0 = 3e8
MIL_TO_M = 25.4e-6

# Cell 4 — Load Geometry

In [4]:
df = pd.read_csv(csv_path)

key = "SIMU_INDEX" if "SIMU_INDEX" in df.columns else "simu_index"
row = df[df[key] == simu_index].iloc[0]

print(row)

simu_index          1.027490e+05
TDIEL               7.666370e+01
TMET                3.303990e+00
XWIDTH              1.425490e+04
YWIDTH              1.462700e+04
CONDUCTIVITY        4.593110e+07
PERMITTIVITY        5.054890e+00
LOSSTANGENT         6.019600e-03
A1_VIARADIUS        5.000000e+00
A1_ANTIPADRADIUS    2.000000e+01
A1_VIAPITCH         4.000000e+01
A1_XCENTER          1.390000e+04
A1_YCENTER          1.355000e+04
A2_VIARADIUS        7.000000e+00
A2_ANTIPADRADIUS    2.300000e+01
A2_VIAPITCH         4.000000e+01
A2_XCENTER          2.250000e+03
A2_YCENTER          6.700000e+03
Name: 2749, dtype: float64


# Cell 5 — Read .s4p

In [5]:
def read_s4p_s11(filepath):
    with open(filepath, "r") as f:
        lines = f.readlines()

    data = []
    freq_unit = "HZ"

    for line in lines:
        line = line.strip()
        if not line or line.startswith("!"):
            continue

        if line.startswith("#"):
            if "MHZ" in line.upper():
                freq_unit = "MHZ"
            elif "GHZ" in line.upper():
                freq_unit = "GHZ"
            continue

        data.extend(line.split())

    data = np.array(data, dtype=float).reshape(-1, 33)

    freq = data[:, 0]
    if freq_unit == "MHZ":
        freq *= 1e6
    elif freq_unit == "GHZ":
        freq *= 1e9

    s11 = data[:, 1] + 1j * data[:, 2]

    return freq, s11

# Cell 6 — Compute Z11

In [6]:
freq, s11 = read_s4p_s11(file_path)

z11 = Z0 * (1 + s11) / (1 - s11)
zmag = np.abs(z11)

print("Z11 computed")

Z11 computed


# Cell 7 — LC Dip

In [7]:
idx = np.argmin(zmag)

f_dip = freq[idx]
z_dip = zmag[idx]

print(f"LC dip = {f_dip/1e6:.2f} MHz")

LC dip = 34.00 MHz


# Cell 8 — Ceff

In [8]:
a = row["XWIDTH"] * MIL_TO_M
b = row["YWIDTH"] * MIL_TO_M
d = row["TDIEL"] * MIL_TO_M
er = row["PERMITTIVITY"]

Ceff = 2 * EPS0 * er * a * b / d

print(f"Ceff = {Ceff*1e9:.3f} nF")

Ceff = 6.184 nF


# Cell 9 — Leff

In [9]:
Leff = 1 / ((2*np.pi*f_dip)**2 * Ceff)

print(f"Leff = {Leff*1e9:.3f} nH")

Leff = 3.544 nH


# Cell 10 — RLC Model

In [10]:
R = 0.03

w = 2*np.pi*freq
Z_rlc = R + 1j*w*Leff + 1/(1j*w*Ceff)
zmag_rlc = np.abs(Z_rlc)

# Cell 11 — Cavity Modes

In [11]:
v = C0 / np.sqrt(er)

modes = []

for m in range(6):
    for n in range(6):
        if m == 0 and n == 0:
            continue
        
        fmn = 0.5 * v * np.sqrt((m/a)**2 + (n/b)**2)
        
        if fmn < freq.max():
            modes.append((m, n, fmn))

modes = sorted(modes, key=lambda x: x[2])

print("First 5 modes:")
for m,n,f in modes[:5]:
    print(f"({m},{n}) → {f/1e6:.2f} MHz")

First 5 modes:
(0,1) → 179.58 MHz
(1,0) → 184.26 MHz
(1,1) → 257.29 MHz
(0,2) → 359.15 MHz
(2,0) → 368.53 MHz


# Cell 12 — Peak Detection

In [12]:
peaks, _ = find_peaks(zmag, prominence=0.5)

print("Number of peaks:", len(peaks))

Number of peaks: 28


# Cell 13 — Plot (Interactive)

In [13]:
fig = go.Figure()

# Simulation
fig.add_trace(go.Scatter(
    x=freq/1e6,
    y=zmag,
    mode="lines",
    name="Z11 Sim.",
    line=dict(color="black", width=2)
))

# RLC
fig.add_trace(go.Scatter(
    x=freq/1e6,
    y=zmag_rlc,
    mode="lines",
    name="RLC model",
    line=dict(color="red", dash="dash")
))

# LC dip
fig.add_trace(go.Scatter(
    x=[f_dip/1e6],
    y=[z_dip],
    mode="markers+text",
    marker=dict(color="green", size=10),
    text=[f"{f_dip/1e6:.2f} MHz"],
    textposition="bottom center",
    name="LC dip"
))

# Peaks
fig.add_trace(go.Scatter(
    x=freq[peaks]/1e6,
    y=zmag[peaks],
    mode="markers",
    marker=dict(color="orange", size=6),
    name="Peaks"
))

# Cavity modes
for m, n, fmn in modes[:10]:
    fig.add_vline(x=fmn/1e6, line_dash="dot", line_color="gray")

fig.update_layout(
    title=f"simu_{simu_index}",
    hovermode="x unified",
    template="plotly_white",
    xaxis=dict(
        title="Frequency (MHz)",
        type="log",
        showspikes=True,
        spikemode="across"
    ),
    yaxis=dict(
        title="|Z11| (Ω)",
        type="log"
    ),
    width=1000,
    height=600
)

fig.show()